In [ ]:
# 필요한 라이브러리 설치 및 임포트
!pip install pandas

import pandas as pd
import datetime

---

# [Step 2] 데이터 전처리

#### **<span style="color:blue">[2-0] air_21, air_22, weather_21, weather_22 데이터 로딩</span>**

In [ ]:
# 데이터 로딩
from pathlib import Path

base_dir = Path.cwd().resolve()
if not (base_dir / 'data').exists():
    base_dir = base_dir.parent

data_dir = base_dir / 'data'
artifacts_dir = base_dir / 'artifacts'
artifacts_dir.mkdir(parents=True, exist_ok=True)

air_21 = pd.read_csv(data_dir / 'air_2021.csv')
air_22 = pd.read_csv(data_dir / 'air_2022.csv')
weather_21 = pd.read_csv(data_dir / 'weather_2021.csv', encoding='cp949')
weather_22 = pd.read_csv(data_dir / 'weather_2022.csv', encoding='cp949')

print(air_21.shape, air_22.shape, weather_21.shape, weather_22.shape)
air_21.head()


#### **<span style="color:blue">[2-1] air_21, air_22 의 '측정일시'를 활용하여 'time' 변수 생성</span>**

* air_21, air_22 의 '측정일시'를 활용하여 'time'변수 생성
* time 변수를 to_datetime으로 데이터 타입 변경
* 참고: 미세먼지 데이터는 1시-24시, 날씨 데이터는 0시-23시로 구성되어 있습니다. [2-5]에서 미세먼지와 날씨 데이터를 time 기준으로 합치려면 날씨 기준이 동일해야 합니다. 미세먼지 데이터에서 time 변수 생성 시 이를 미리 고려(예:측정일시 값 -1)하세요.

In [ ]:
# 아래에 필요한 코드를 작성하고 결과를 확인합니다.
for air_df in [air_21, air_22]:
    ts = air_df['측정일시'].astype(str).str.zfill(10)
    air_df['time'] = pd.to_datetime(ts.str[:8], format='%Y%m%d') + pd.to_timedelta(
        ts.str[8:].astype(int) - 1,
        unit='h'
    )

print(air_21[['측정일시', 'time']].head())
print(air_22[['측정일시', 'time']].head())


---

#### **<span style="color:blue">[2-2] weather_21, weather_22 의 '일시'를 활용하여 'time' 변수 생성</span>**

* weather_21, weather_22 의 '일시'를 활용하여 'time'변수 생성
* time 변수를 to_datetime으로 데이터 타입 변경

In [ ]:
# 아래에 필요한 코드를 작성하고 결과를 확인합니다.
for weather_df in [weather_21, weather_22]:
    weather_df['time'] = pd.to_datetime(weather_df['일시'])

print(weather_21[['일시', 'time']].head())
print(weather_22[['일시', 'time']].head())


---

#### **<span style="color:blue">[2-3] 'time' 기준으로 데이터 합치기</span>**

* 미세먼지 데이터와 날씨 데이터를 'time' 기준으로 합쳐보세요.
* df_21에는 'time' 기준으로 21년도 미세먼지, 날씨 데이터를 합쳐보세요.
* df_22에는 'time' 기준으로 22년도 미세먼지, 날씨 데이터를 합쳐보세요.

In [ ]:
# 아래에 필요한 코드를 작성하고 결과를 확인합니다.
df_21 = pd.merge(air_21, weather_21, on='time', how='inner')
df_22 = pd.merge(air_22, weather_22, on='time', how='inner')

print(df_21.shape, df_22.shape)
df_21.head()


---

#### **<span style="color:blue">[2-4] 사용하지 않을 변수 제거</span>**

* 머신러닝에 사용하지 않을 변수들을 제거해줍니다.
* df_21, df_22에 사용할 변수들만 넣어보세요.

In [ ]:
# df_21, df_22에 사용할 변수들만 할당
use_cols = [
    'time', 'SO2', 'CO', 'O3', 'NO2', 'PM10', 'PM25',
    '기온(°C)', '강수량(mm)', '풍속(m/s)', '풍향(16방위)', '습도(%)',
    '증기압(hPa)', '이슬점온도(°C)', '현지기압(hPa)', '해면기압(hPa)',
    '일조(hr)', '일사(MJ/m2)', '적설(cm)', '3시간신적설(cm)',
    '전운량(10분위)', '중하층운량(10분위)', '최저운고(100m )', '시정(10m)',
    '지면온도(°C)', '5cm 지중온도(°C)', '10cm 지중온도(°C)',
    '20cm 지중온도(°C)', '30cm 지중온도(°C)'
]

df_21 = df_21[use_cols].copy()
df_22 = df_22[use_cols].copy()

print(df_21.columns.tolist())


In [ ]:
# time 변수를 index로 세팅
df_21 = df_21.set_index('time').sort_index()
df_22 = df_22.set_index('time').sort_index()

df_21.head()


---

#### **<span style="color:blue">[2-5] 변수들의 결측치 처리</span>**

In [ ]:
# df_21, df_22의 결측치 확인
print(df_21.isna().sum().sort_values(ascending=False))
print()
print(df_22.isna().sum().sort_values(ascending=False))


In [ ]:
# df_21, df_22의 변수 중 '강수량(mm)'의 결측치를 처리
df_21['강수량(mm)'] = df_21['강수량(mm)'].fillna(0)
df_22['강수량(mm)'] = df_22['강수량(mm)'].fillna(0)

print(df_21['강수량(mm)'].isna().sum(), df_22['강수량(mm)'].isna().sum())


In [ ]:
# df_21, df_22의 남은 결측치를 처리
df_21 = df_21.ffill().bfill()
df_22 = df_22.ffill().bfill()


In [ ]:
# df_21, df_22의 결측치 재확인
print(df_21.isna().sum().sum())
print(df_22.isna().sum().sum())


---

#### **<span style="color:blue">[2-6] 전일 같은 시간 변수 추가</span>**

* 모델링에 유용한 변수로 전일 같은 시간(24시간 전) 미세먼지 농도 변수를 추가합니다.
* 시계열 데이터 처리를 위한 shift 연산을 참고하세요.

In [ ]:
# df_21, df_22의 index(time)를 month, day, hour 로 쪼개기 (year는 필요 없음)
for df in [df_21, df_22]:
    df['month'] = df.index.month
    df['day'] = df.index.day
    df['hour'] = df.index.hour

df_21[['month', 'day', 'hour']].head()


In [ ]:
# df_21, df_22에 전일 같은 시간 미세먼지 농도 변수(PM10_lag1) 추가
# 전일 같은 시간은 24시간 전 입니다.
df_21['PM10_lag1'] = df_21['PM10'].shift(24)
df_22['PM10_lag1'] = df_22['PM10'].shift(24)

df_21[['PM10', 'PM10_lag1']].head(30)


---

#### **<span style="color:blue">[2-7] t+1 시점의 미세먼지 농도 데이터 생성</span>**

* t+1 시점은 1시간 후 입니다.
* t+1 시점의 미세먼지 농도 변수를 생성하세요.
* t+1 시점의 미세먼지 농도는 머신러닝 모델을 통해 예측하려는 y값(target) 입니다.

In [ ]:
# df_21, df_22에 t+1 시점 변수(PM10_1) 추가
df_21['PM10_1'] = df_21['PM10'].shift(-1)
df_22['PM10_1'] = df_22['PM10'].shift(-1)

df_21[['PM10', 'PM10_1']].head()


In [ ]:
# 결측치가 있다면 처리
df_21 = df_21.dropna().copy()
df_22 = df_22.dropna().copy()

print(df_21.shape, df_22.shape)
print(df_21.isna().sum().sum(), df_22.isna().sum().sum())


---

#### **<span style="color:blue">[2-8] train, test 데이터 분리</span>**

* 21년도 데이터(df_21)를 train 데이터로 저장하세요. y 값을 제외하고 train_x로 저장한 후 y 값은 train_y로 저장하세요.
* 22년도 데이터(df_22)를 test 데이터로 저장하세요. y 값을 제외하고 test_x로 저장한 후 y 값은 test_y로 저장하세요.
* 각각의 데이터프레임을 csv 파일로 저장하세요. (train_x.csv / train_y.csv / test_x.csv / test_y.csv)
* y값은 'PM10_1' 즉, t+1 시점의 미세먼지 농도입니다.

In [ ]:
# 아래에 필요한 코드를 작성하고 결과를 확인합니다.
train_x = df_21.drop(columns='PM10_1').copy()
train_y = df_21[['PM10_1']].copy()

test_x = df_22.drop(columns='PM10_1').copy()
test_y = df_22[['PM10_1']].copy()

print(train_x.shape, train_y.shape)
print(test_x.shape, test_y.shape)
train_x.head()


In [ ]:
# 각각의 데이터프레임을 csv 파일로 저장 (train_x.csv / train_y.csv / test_x.csv / test_y.csv)
train_x.to_csv(artifacts_dir / 'train_x.csv', index=False)
train_y.to_csv(artifacts_dir / 'train_y.csv', index=False)
test_x.to_csv(artifacts_dir / 'test_x.csv', index=False)
test_y.to_csv(artifacts_dir / 'test_y.csv', index=False)

print(f'train_x.csv saved to: {artifacts_dir / "train_x.csv"}')
print(f'train_y.csv saved to: {artifacts_dir / "train_y.csv"}')
print(f'test_x.csv saved to: {artifacts_dir / "test_x.csv"}')
print(f'test_y.csv saved to: {artifacts_dir / "test_y.csv"}')
